# Exploratory data analysis

This notebook looks at the synthetic training data, `data/data.csv`. It trains no model and saves no file.
The model is trained by `medilinda_ml.modeling.train_model`.

Start the kernel with `PYTHONPATH` set to `server/src`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from medilinda_ml.columns import DRUG_NAMES, INITIAL_CATEGORICAL_COLUMNS, TARGET_COLUMN
from medilinda_ml.paths import DATA_DIR

sns.set_theme(style="whitegrid")

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_DIR / "data.csv")
df.shape

## 2. Target balance

The target is `causality_assessment_level`. A skewed target is the reason that training uses SMOTE.

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
target_share = df[TARGET_COLUMN].value_counts(normalize=True).round(3)
pd.DataFrame({"rows": target_counts, "share": target_share})

In [ ]:
sns.countplot(data=df, x=TARGET_COLUMN, order=target_counts.index)
plt.title("Rows for each causality assessment level")
plt.show()

## 3. Missing values

A null in a drug column means that the drug was not suspected in that report. A null in `patient_age` means that the report has `patient_date_of_birth`, and the reverse.

In [ ]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
age_or_birth_date_present = df["patient_age"].notna() | df["patient_date_of_birth"].notna()
print("Rows with no age and no date of birth:", int((~age_or_birth_date_present).sum()))

for drug in DRUG_NAMES:
    suspected = df[f"{drug}_suspected"]
    dose_missing = df[f"{drug}_dose_amount"].isnull()
    print(f"{drug}: suspected {int(suspected.sum())}, dose missing {int(dose_missing.sum())}, "
          f"dose missing for a suspected drug {int((suspected & dose_missing).sum())}")

## 4. Numerical columns

In [ ]:
numerical_columns = ["patient_age", "patient_weight_kg", "patient_height_cm"]
df[numerical_columns].describe().round(1)

In [ ]:
figure, axes = plt.subplots(1, len(numerical_columns), figsize=(15, 4))
for axis, column in zip(axes, numerical_columns):
    sns.boxplot(data=df, x=TARGET_COLUMN, y=column, order=target_counts.index, ax=axis)
    axis.set_title(column)
plt.tight_layout()
plt.show()

## 5. Categorical columns by target

Each cell is the share of rows of one target level that have the value. Each column of the heat map sums to one.

In [ ]:
for column in INITIAL_CATEGORICAL_COLUMNS:
    share_by_target = pd.crosstab(df[column], df[TARGET_COLUMN], normalize="columns")
    plt.figure(figsize=(6, 0.5 * len(share_by_target) + 1.5))
    sns.heatmap(share_by_target, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
    plt.title(column)
    plt.show()